# SIAT data collection (OpenAlex)

Resolves OpenAlex author IDs for SIAT research and teaching faculty, then downloads their works into `siat_data/`.
The dashboard (`app.py`) reads these files, so this notebook contains no visualisations.

**Run order:** the setup, people and resolve cells, then review the unresolved list, fill in `MANUAL`, then run the roster and fetch cells.
Re-run the fetch cell with `refresh=True` to update the data.

Outputs: `siat_data/roster.json` (name to author IDs), `siat_data/groups.json` (name to group), `siat_data/works.json` (cached works, including institutions, countries and primary topics).

An OpenAlex API key is required (free: https://openalex.org/settings/api). Set it as the `OPENALEX_API_KEY` environment variable, or paste it into the setup cell.

In [1]:
# %pip install requests pandas
import os, json, time, unicodedata
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict
import requests, pandas as pd
from IPython.display import display

MAILTO = "juan@alperin.ca"
API_KEY = os.environ.get("OPENALEX_API_KEY")     # or: API_KEY = "your-key-here"
DATA = Path("siat_data"); DATA.mkdir(exist_ok=True)
BASE = "https://api.openalex.org"

def oa_get(path, **params):
    if not API_KEY:
        raise RuntimeError("Set OPENALEX_API_KEY (free key at https://openalex.org/settings/api)")
    params = {k: v for k, v in params.items() if v is not None}
    params["mailto"] = MAILTO
    headers = {"Authorization": f"Bearer {API_KEY}"}
    for attempt in range(5):
        r = requests.get(f"{BASE}/{path}", params=params, headers=headers, timeout=60)
        if r.status_code >= 500:
            time.sleep(2 ** attempt); continue
        if r.status_code == 429:
            raise RuntimeError("OpenAlex budget or rate limit hit; check budget(). Cached data is still usable.")
        if r.status_code == 401:
            raise RuntimeError("401: OpenAlex rejected the key (missing, mistyped, or rotated).")
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

def budget():
    """Remaining daily API budget."""
    return requests.get(f"{BASE}/rate-limit", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=30).json()

short = lambda i: i.rsplit("/", 1)[-1]
full = lambda i: i if i.startswith("http") else "https://openalex.org/" + i

def norm(s):
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c)).lower().replace("-", " ")

budget()

{'api_key': 'pCa...RNU',
 'is_grandfathered': False,
 'rate_limit': {'daily_budget_usd': 1,
  'daily_used_usd': 0.1813,
  'daily_remaining_usd': 0.8187,
  'prepaid_balance_usd': 0,
  'prepaid_remaining_usd': 0,
  'prepaid_expires_at': None,
  'resets_at': '2026-11-14T00:00:00.000Z',
  'resets_in_seconds': 3168,
  'endpoint_costs_usd': {'singleton': 0,
   'list': 0.0001,
   'search': 0.001,
   'search_rerank': 0.002,
   'rerank_extra': 0.001,
   'content': 0.01,
   'semantic': 0.001,
   'text': 0.01},
  'credits_limit': 10000,
  'credits_used': 1813,
  'credits_remaining': 8187,
  'onetime_credits_balance': 0,
  'onetime_credits_remaining': 0,
  'onetime_credits_expires_at': None,
  'credit_costs': {'singleton': 0,
   'list': 1,
   'search': 10,
   'search_rerank': 20,
   'rerank_extra': 10,
   'content': 100,
   'semantic': 10,
   'text': 100}}}

In [2]:
# name -> extra search variants
ROSTER = {
    "Gabriela Aceves-Sepulveda": ["Gabriela Aceves Sepulveda"],
    "Juan Pablo Alperin": ["Juan Alperin"],
    "Alissa Antle": [],
    "Sheelagh Carpendale": [],
    "Parmit Chilana": [],
    "Jon Corbett": ["Jon Corbett Okanagan"],
    "Steve DiPaola": ["Stephen DiPaola"],
    "Halil Erhan": ["Halil I. Erhan"],
    "Brian Fisher": ["Brian D. Fisher"],
    "Marek Hatala": [],
    "Kate Hennessy": ["Katherine Hennessy"],
    "Alireza Karduni": [],
    "Sylvain Moreno": [],
    "Carman Neustaedter": [],
    "Will Odom": ["William Odom", "William T. Odom"],
    "Philippe Pasquier": [],
    "Niranjan Rajah": [],
    "Bernhard Riecke": ["Bernhard E. Riecke"],
    "Gillian Russell": [],
    "Thecla Schiphorst": [],
    "Chris Shaw": ["Christopher Shaw", "Christopher D. Shaw"],
    "Wolfgang Stuerzlinger": ["Wolfgang Stürzlinger"],
    "Palashi Vaghela": [],
    "Ron Wakkary": [],
    "Ö. Nilay Yalçin": ["Nilay Yalcin", "Özge Nilay Yalçın"],
}

TEACHING = {
    "Susan Clements-Vivian": ["Susan Clements Vivian"],
    "Michael Filimowicz": [],
    "Chantal Gibson": [],
    "Andrew Hawryshkewich": [],
    "Mark Nazemi": [],
    "Mauve Pagé": ["Mauve Page"],
    "Helmine Serban": [],
    "Eric Yang": ["Eric Y. Yang"],
    "Ken Zupan": ["Kenneth Zupan"],
}

GROUPS = {**{n: "Research faculty" for n in ROSTER}, **{n: "Teaching faculty" for n in TEACHING}}
ALL_PEOPLE = {**ROSTER, **TEACHING}
print(len(ROSTER), "research,", len(TEACHING), "teaching")

25 research, 9 teaching


## Resolve author IDs

In [3]:
def find_sfu_id():
    res = oa_get("institutions", search="Simon Fraser University", filter="country_code:CA", **{"per-page": 3})
    return res["results"][0]["id"]

SFU_ID = find_sfu_id()

def at_sfu(a):
    ids = {i["id"] for i in a.get("last_known_institutions") or []}
    ids |= {x["institution"]["id"] for x in a.get("affiliations") or []}
    return SFU_ID in ids

def name_ok(a, variants):
    names = [norm(a["display_name"])] + [norm(x) for x in a.get("display_name_alternatives") or []]
    lasts = {norm(v).split()[-1] for v in variants}
    return any(l in n.split() for l in lasts for n in names)

def cand_row(name, status, a):
    return dict(
        roster_name=name, status=status, id=short(a["id"]), oa_name=a["display_name"],
        at_sfu=at_sfu(a), works=a["works_count"], cites=a["cited_by_count"],
        last_inst="; ".join(i["display_name"] for i in a.get("last_known_institutions") or []),
        topics="; ".join(t["display_name"] for t in (a.get("topics") or [])[:3]),
    )

def resolve_roster():
    resolved, rows = {}, []
    for name, extra in ALL_PEOPLE.items():
        variants = [name] + extra
        seen = {}
        for q in variants:
            for a in oa_get("authors", search=q, **{"per-page": 15})["results"]:
                seen[a["id"]] = a
        cands = [a for a in seen.values() if name_ok(a, variants)]
        sfu = [a for a in cands if at_sfu(a)]
        if len(sfu) == 1:
            status = "matched"; resolved[name] = [short(sfu[0]["id"])]
        elif len(sfu) > 1:
            status = "ambiguous: multiple SFU profiles"
        elif cands:
            status = "no SFU-affiliated profile"
        else:
            status = "not found"
        shown = sfu if sfu else sorted(cands, key=lambda a: -a["works_count"])[:6]
        rows += [cand_row(name, status, a) for a in shown] or [dict(roster_name=name, status=status)]
    return resolved, pd.DataFrame(rows)

resolved, cand_df = resolve_roster()
cand_df["group"] = cand_df.roster_name.map(GROUPS)
review = cand_df[cand_df.status != "matched"]
print("Matched:", len(resolved), "| Need manual review:", review.roster_name.nunique())
print(sorted(review.roster_name.unique()))
pd.set_option("display.max_colwidth", 60)
display(review)

Matched: 27 | Need manual review: 7
['Ken Zupan', 'Mark Nazemi', 'Mauve Pagé', 'Niranjan Rajah', 'Sheelagh Carpendale', 'Susan Clements-Vivian', 'Will Odom']


,roster_name,status,id,oa_name,at_sfu,works,cites,last_inst,topics,group
3,Sheelagh Carpendale,ambiguous: multiple SFU profiles,A5012561411,Sheelagh Carpendale,True,441,13262,Simon Fraser University; University of Victoria,Data Visualization and Analytics; Interactive and Immers...,Research faculty
4,Sheelagh Carpendale,ambiguous: multiple SFU profiles,A5018647103,M. Sheelagh T. Carpendale,True,42,1517,University of Calgary,Data Visualization and Analytics; Computer Graphics and ...,Research faculty
15,Will Odom,ambiguous: multiple SFU profiles,A5056121536,Will Odom,True,16,123,,Innovative Human-Technology Interaction; Palliative Care...,Research faculty
16,Will Odom,ambiguous: multiple SFU profiles,A5073024818,William Odom,True,182,5716,Simon Fraser University,Innovative Human-Technology Interaction; Interactive and...,Research faculty
18,Niranjan Rajah,no SFU-affiliated profile,A5030785945,Niranjan Rajah,False,3,2,,"Philosophy and Theoretical Science; Neuroethics, Human E...",Research faculty
27,Susan Clements-Vivian,no SFU-affiliated profile,A5089195430,Susan Clements-Vivian,False,1,0,,"Visual Culture and Art Theory; Art, Technology, and Cult...",Teaching faculty
31,Mark Nazemi,ambiguous: multiple SFU profiles,A5055778658,Mark Nazemi,True,11,68,Simon Fraser University,Virtual Reality Applications and Impacts; Pediatric Pain...,Teaching faculty
32,Mark Nazemi,ambiguous: multiple SFU profiles,A5009055178,Mehdi Mark Nazemi,True,6,13,Ferdowsi University of Mashhad,Music Therapy and Health; Noise Effects and Management; ...,Teaching faculty
33,Mauve Pagé,no SFU-affiliated profile,A5053086840,MariÈve Mauve Pagé,False,2,1,,Antibiotic Resistance in Bacteria; Influenza Virus Resea...,Teaching faculty
36,Ken Zupan,no SFU-affiliated profile,A5054611373,Ken Zupan,False,1,3,University of Southern Denmark,"Innovative Human-Technology Interaction; Art, Technology...",Teaching faculty


## Manual overrides and final roster

Put IDs you have checked here. A person can have several IDs if OpenAlex split their profile. Manual entries override automatic matches.

In [4]:
MANUAL = {
    # "Sheelagh Carpendale" was listed twice (a two-ID entry, then a one-ID entry). Python keeps the last one,
    # so only A5012561411 is used. To merge her split profiles instead, use: ["A5012561411", "A5018647103"]
    "Sheelagh Carpendale": ["A5012561411", "A5018647103"],
    "Will Odom": ["A5056121536"],
    "Mark Nazemi": ["A5055778658"],
    "Mauve Pagé": ["A5053086840"],
    "Susan Clements-Vivian": ["A5089195430"],
}

roster = {**resolved, **{k: [short(i) for i in (v if isinstance(v, list) else [v])] for k, v in MANUAL.items()}}
missing = [n for n in ALL_PEOPLE if n not in roster]
print("Still unresolved (excluded from dashboard):", missing)
(DATA / "roster.json").write_text(json.dumps(roster, indent=2))
(DATA / "groups.json").write_text(json.dumps({n: GROUPS[n] for n in roster}, indent=2))

Still unresolved (excluded from dashboard): ['Niranjan Rajah', 'Ken Zupan']


1318

## Fetch works

Cached for 7 days and invalidated automatically if the roster IDs change. Use `refresh=True` to force a new pull. Nothing is written until a full fetch completes.

In [5]:
SELECT = "id,doi,title,publication_year,type,cited_by_count,authorships,referenced_works,primary_topic"
EXCLUDE_TYPES = {"paratext", "erratum", "retraction"}
CACHE = DATA / "works.json"
SCHEMA = 2   # bump when the stored fields change; forces a refetch

def slim(w):
    """Keep only what the dashboard uses (author, institutions, countries, primary topic). Keeps the cache small."""
    lvl = lambda d: {"id": (d or {}).get("id"), "display_name": (d or {}).get("display_name")}
    pt = w.get("primary_topic")
    if pt:
        pt = {**lvl(pt), **{k: lvl(pt.get(k)) for k in ("subfield", "field", "domain")}}
    return {
        "id": w["id"], "doi": w.get("doi"), "title": w.get("title"), "type": w.get("type"),
        "publication_year": w.get("publication_year"), "cited_by_count": w.get("cited_by_count"),
        "referenced_works": w.get("referenced_works") or [], "primary_topic": pt,
        "authorships": [{
            "author": lvl(au.get("author")),
            "institutions": [{k: i.get(k) for k in ("id", "display_name", "country_code", "type")}
                             for i in au.get("institutions") or []],
            "countries": au.get("countries") or [],
        } for au in w.get("authorships") or []],
    }

def fetch_works(roster, max_age_days=7, refresh=False):
    ids = sorted({i for v in roster.values() for i in v})
    if CACHE.exists() and not refresh:
        blob = json.loads(CACHE.read_text())
        fresh = datetime.fromisoformat(blob["fetched"]) > datetime.now() - timedelta(days=max_age_days)
        if blob["ids"] == ids and blob.get("schema") == SCHEMA and fresh:
            print("Using cache from", blob["fetched"][:16])
            return blob["works"]
    works = {}
    for k in range(0, len(ids), 25):
        cursor = "*"
        while cursor:
            res = oa_get("works", filter="author.id:" + "|".join(ids[k:k+25]),
                         select=SELECT, cursor=cursor, **{"per-page": 200})
            for w in res["results"]:
                if w["type"] not in EXCLUDE_TYPES:
                    works[w["id"]] = slim(w)
            cursor = res["meta"].get("next_cursor")
        print(f"  fetched author batch {k // 25 + 1}: {len(works)} works so far")
    CACHE.write_text(json.dumps(dict(fetched=datetime.now().isoformat(), ids=ids, schema=SCHEMA, works=works)))
    return works

works = fetch_works(roster, refresh=False)
print(len(works), "works saved to", CACHE)
n_topic = sum(1 for w in works.values() if w.get("primary_topic"))
n_aff = sum(1 for w in works.values() for a in w["authorships"] if a["institutions"])
n_all = sum(len(w["authorships"]) for w in works.values())
print(f"{n_topic}/{len(works)} works have a primary topic; {n_aff}/{n_all} authorships have institution data")

Using cache from 2026-11-13T16:04
3647 works saved to siat_data/works.json
3605/3647 works have a primary topic; 11650/18905 authorships have institution data


## Sanity check

Works per person, to spot split profiles or empty results before opening the dashboard. This is a table only.

In [6]:
id2m = {full(i): n for n, ids in roster.items() for i in ids}
counts = defaultdict(int)
for w in works.values():
    for m in {id2m[a["author"]["id"]] for a in w["authorships"] if a.get("author") and a["author"].get("id") in id2m}:
        counts[m] += 1
check = pd.DataFrame([dict(person=n, group=GROUPS[n], works=counts.get(n, 0)) for n in roster]).sort_values("works")
display(check)
print("Run the dashboard with:  python app.py")

,person,group,works
31,Susan Clements-Vivian,Teaching faculty,1
30,Mauve Pagé,Teaching faculty,2
25,Helmine Serban,Teaching faculty,3
23,Chantal Gibson,Teaching faculty,3
4,Jon Corbett,Research faculty,5
24,Andrew Hawryshkewich,Teaching faculty,5
28,Will Odom,Research faculty,8
29,Mark Nazemi,Teaching faculty,9
19,Palashi Vaghela,Research faculty,9
15,Gillian Russell,Research faculty,15


Run the dashboard with:  python app.py


## Review a person's collaborations

Use this to check where an institution in the dashboard comes from. It works on the cached `works.json` (no API calls).
Institutions are taken from each **co-author's** affiliation on each work, as recorded by OpenAlex at publication time.

In [7]:
def person_works(person):
    ids = {full(i) for i in roster[person]}
    return [w for w in works.values() if any(a["author"]["id"] in ids for a in w["authorships"])], ids

def collab_rows(person, since=None):
    """One row per (work, external co-author, institution). Other roster members are skipped."""
    mine, ids = person_works(person)
    everyone = {full(i) for v in roster.values() for i in v}
    rows = []
    for w in mine:
        if since and (w["publication_year"] or 0) < since: continue
        for a in w["authorships"]:
            if not a["author"].get("id") or a["author"]["id"] in everyone: continue   # skip unresolved authors (no OpenAlex ID)
            for i in a["institutions"] or [{}]:
                rows.append(dict(year=w["publication_year"], title=w["title"], work_id=short(w["id"]), doi=w.get("doi"),
                                 coauthor=a["author"]["display_name"], coauthor_id=short(a["author"]["id"]),
                                 institution=i.get("display_name") or "(none listed)", country=i.get("country_code")))
    return pd.DataFrame(rows)

def institution_summary(person, since=None, top=40):
    """All external institutions for one person, strongest first, with example co-authors."""
    df = collab_rows(person, since)
    g = df.groupby(["institution", "country"], dropna=False)
    out = g.agg(works=("work_id", "nunique"), coauthors=("coauthor_id", "nunique"),
                example_coauthors=("coauthor", lambda s: "; ".join(list(dict.fromkeys(s))[:3]))).reset_index()
    return out.sort_values(["works", "coauthors"], ascending=False).head(top)

def why_institution(person, inst_contains, since=None):
    """Every work and co-author behind one institution. Open the DOI to check the paper."""
    df = collab_rows(person, since)
    hit = df[df.institution.str.contains(inst_contains, case=False, na=False)]
    return hit.sort_values(["coauthor", "year"], ascending=[True, False])[
        ["coauthor", "institution", "year", "title", "doi", "work_id", "coauthor_id"]]

def raw_affiliations(work_ids, coauthor_ids):
    """Fetch the raw affiliation text OpenAlex used for given co-authors on given works (live API call)."""
    res = oa_get("works", filter="openalex:" + "|".join(work_ids), select="id,title,authorships", **{"per-page": 50})
    want = {full(i) for i in coauthor_ids}
    rows = []
    for w in res["results"]:
        for a in w["authorships"]:
            if a["author"]["id"] in want:
                rows.append(dict(work=short(w["id"]), title=w["title"], coauthor=a["author"]["display_name"],
                                 raw_affiliation=" | ".join(a.get("raw_affiliation_strings") or []) or "(none)",
                                 matched=[i["display_name"] for i in a["institutions"]]))
    return pd.DataFrame(rows)

ME = "Juan Pablo Alperin"
pd.set_option("display.max_colwidth", 80, "display.max_rows", 200)
display(institution_summary(ME))

,institution,country,works,coauthors,example_coauthors
0,(none listed),NaN,77,274,Gustavo E. Fischman; John M. Willinsky; Lesley A. Schimanski
71,Simon Fraser University,CA,62,31,Lisa Matthias; Erin C. McKiernan; Lesley A. Schimanski
121,University of Ottawa,CA,21,23,Stefanie Haustein; Kelly Denise Cobey; Jamie Brehaut
74,Stanford University,US,17,7,Saurabh Khanna; Jon Ball; John M. Willinsky
136,Université de Montréal,CA,14,12,Vincent Larivière; Simon van Bellen; Marc‐André Simard
101,University of British Columbia,CA,14,5,Esteban Morales; Clare L. Ardern; Alice Fleerackers
133,University of Vermont,US,12,5,Erin C. McKiernan; Lesley A. Schimanski; Carol Muñoz Nieves
87,Universidad Nacional Autónoma de México,MX,12,1,Erin C. McKiernan
69,San Francisco State University,US,10,2,Laura L. Moorhead; Kaylee Fagan
137,Université du Québec à Montréal,CA,9,5,Vincent Larivière; Stefanie Haustein; Fei Shu


In [8]:
# Which works and co-authors put an institution on your list?
hits = why_institution(ME, "British Columbia")
display(hits)

# Optional: see the raw affiliation text OpenAlex used for those co-authors (one live API call)
# display(raw_affiliations(hits.work_id.unique()[:50], hits.coauthor_id.unique()))

,coauthor,institution,year,title,doi,work_id,coauthor_id
313,Alice Fleerackers,University of British Columbia,2025.0,From impact metrics and open science to communicating research: Journalists’...,https://doi.org/10.1371/journal.pone.0309274,W4406861737,A5070557231
189,Alice Fleerackers,University of British Columbia,2024.0,Second-order citations in altmetrics: A case study analyzing the audiences o...,https://doi.org/10.1162/qss_a_00298,W4392454393,A5070557231
214,Alice Fleerackers,University of British Columbia,2024.0,Bridging science communication and open science—Working inclusively toward t...,https://doi.org/10.3389/fcomm.2024.1473268,W4402989579,A5070557231
269,Alice Fleerackers,University of British Columbia,2024.0,The impact of COVID-19 on the debate on open science: a qualitative analysis...,https://doi.org/10.1057/s41599-024-03804-w,W4403065771,A5070557231
560,Alice Fleerackers,University of British Columbia,2024.0,Stark Decline in Journalists’ Use of Preprints Postpandemic,https://doi.org/10.1177/10755470241285405,W4403334593,A5070557231
70,Clare L. Ardern,University of British Columbia,2023.0,Community consensus on core open science practices to monitor in biomedicine,https://doi.org/10.1371/journal.pbio.3001949,W4317866649,A5020960452
917,Didegah,University of British Columbia,2019.0,"Text data for comparing the language used in Wikipedia, the DiabeticConnect ...",https://doi.org/10.7910/dvn/x27dse,W4399003852,A5098824447
40,Esteban Morales,University of British Columbia,2022.0,"The role of collegiality in academic review, promotion, and tenure",https://doi.org/10.1371/journal.pone.0265506,W4226317078,A5083084802
237,Esteban Morales,University of British Columbia,2022.0,Using social annotation to construct knowledge with others: A case study acr...,https://doi.org/10.12688/f1000research.109525.2,W4226490301,A5083084802
713,Esteban Morales,University of British Columbia,2022.0,Knowledge Construction Activities in the Online Social Annotations of Three ...,https://doi.org/10.7910/dvn/svdfcf,W4398810579,A5083084802
